In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Knapsack"
weight_col = "weight"
value_col = "value"

df = pd.read_excel(file_path, sheet_name=sheet_name)
weights = df[weight_col].astype(int).to_list()
values = df[value_col].astype(float).to_list()

# ========= 2) 参数模板 =========
params = {
    "capacity": 50  # int: 背包容量
}

n = len(weights)
dp = np.zeros((n + 1, params["capacity"] + 1))
for i in range(1, n + 1):
    w, v = weights[i - 1], values[i - 1]
    for c in range(params["capacity"] + 1):
        dp[i, c] = dp[i - 1, c]
        if c >= w:
            dp[i, c] = max(dp[i, c], dp[i - 1, c - w] + v)

print("最优值:", dp[n, params["capacity"]])


In [ ]:
"""
动态优化模型

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "动态优化模型.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
PERIODS = 4  # TODO: 请填写[阶段数]，说明：正整数。
DEMAND = [2, 3, 1, 4]  # TODO: 请填写[各阶段需求]，说明：长度等于阶段数。
MAX_INVENTORY = 10  # TODO: 请填写[最大库存]，说明：非负整数。
MAX_ORDER = 10  # TODO: 请填写[最大订货量]，说明：非负整数。
INITIAL_STOCK = 0  # TODO: 请填写[初始库存]，说明：0 到 MAX_INVENTORY。
ORDER_COST = 2.0  # TODO: 请填写[单位订货成本]，说明：非负数。
HOLDING_COST = 0.5  # TODO: 请填写[单位库存成本]，说明：非负数。



REQUIRES_DATA = False  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    # 示例：有限期库存动态规划，状态为期初库存，动作为订货量。
    dp = np.full((PERIODS + 1, MAX_INVENTORY + 1), np.inf)
    policy = np.zeros((PERIODS, MAX_INVENTORY + 1), dtype=int)
    dp[PERIODS, :] = 0
    for t in range(PERIODS - 1, -1, -1):
        for stock in range(MAX_INVENTORY + 1):
            for order in range(MAX_ORDER + 1):
                next_stock = stock + order - DEMAND[t]
                if 0 <= next_stock <= MAX_INVENTORY:
                    cost = ORDER_COST * order + HOLDING_COST * next_stock + dp[t + 1, next_stock]
                    if cost < dp[t, stock]:
                        dp[t, stock] = cost
                        policy[t, stock] = order
    print("最小总成本:", dp[0, INITIAL_STOCK])
    print("各期订货策略:", [policy[t, INITIAL_STOCK if t == 0 else 0] for t in range(PERIODS)])


if __name__ == "__main__":
    df = load_data()
    run_model(df)


# 动态优化模型

## 输入说明

- 数据文件：默认读取脚本同目录下的 `data.csv`，也可以在代码顶部把 `DATA_FILE` 改为 `.xlsx` 或绝对路径。
- 数据格式：一般要求“一行一个样本/时刻/方案，一列一个变量/指标”。具体列名需要在代码顶部的 `TODO` 参数区填写。
- 示例：若模型需要特征 `特征1、特征2` 和目标列 `y`，表格可整理为：

| 特征1 | 特征2 | y |
|---:|---:|---:|
| 1.2 | 3.4 | 8.1 |
| 2.0 | 2.8 | 9.0 |

## 输出说明

- 控制台会打印核心结果，例如模型参数、评价指标、最优解、排名或预测值。
- 默认结果保存到代码顶部 `OUTPUT_FILE` 指定的文件。
- 若模型包含图形分析，会额外输出图片文件，例如箱型图 `boxplot.png`。

## 原理通俗解释

动态优化 的核心思想是：先把实际问题抽象成可计算的数据结构，再用对应的数学规则寻找“预测值、分类结果、综合得分或最优方案”。代码中已经保留主要计算流程，比赛时重点是把题目数据整理成表格，并把 TODO 参数替换为题目含义一致的列名和约束。

## 适用场景

多阶段决策，如库存、路径、生产计划。

## 局限性

状态空间过大时会出现维数灾难。

## 使用提示

- 运行前先检查缺失值、异常值和量纲；很多模型对数据尺度敏感。
- 所有 `TODO` 都应结合题目背景填写，不要直接使用示例列名。
- 建模论文中建议同时写明参数来源，例如权重来自 AHP/熵权法，预测步数来自题目要求。
